# 02. Data Preprocessing

## 1. Objective
This notebook prepares the FDIC bank-level quarterly panel
for subsequent feature engineering and predictive modeling.

## 2. Main Tasks
- Load and inspect raw financial and failure data.
- Clean and standardize variables.
- Construct the bank-quarter panel.
- Create the four-quarter-ahead failure label.
- Handle merger-related censoring.
- Validate and export the processed dataset.

## 3. Output
The final processed dataset will be saved in Parquet format
for use in the Feature Engineering and Modeling notebooks.


In [1]:
# 0. Imports and project paths (portable: no machine-specific path)
from pathlib import Path
import polars as pl

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "processed" / "v3").exists():
            return candidate
    raise FileNotFoundError(
        "Cannot locate data/processed/v3. Open the project folder in VS Code "
        "and run this notebook from inside it."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
V3_DIR = PROCESSED_DIR / "v3"

print("Project root folder:", PROJECT_ROOT.name)
for name, folder in [("Raw", RAW_DIR), ("Processed", PROCESSED_DIR), ("V3", V3_DIR)]:
    print(f"{name} directory exists:", folder.exists())
    if not folder.exists():
        raise FileNotFoundError(f"{name} directory not found: {folder.name}")

Project root folder: BFT65B_Final_Project
Raw directory exists: True
Processed directory exists: True
V3 directory exists: True


In [12]:
# 1. List processed datasets
print("=== PROCESSED DATA ===")

for file in sorted((PROCESSED_DIR / "v3").glob("*.parquet")):
    size_mb = file.stat().st_size / (1024 ** 2)
    print(f"{file.name} | {size_mb:.2f} MB")

# 2. Check the feature dataset
feature_path = PROCESSED_DIR / "v3" / "bank_quarter_features_v3.parquet"

if feature_path.exists():
    feature_data = pl.read_parquet(feature_path)

    print("\n=== FEATURE DATASET CHECK ===")
    print("Shape:", feature_data.shape)
    print("Columns:", feature_data.columns)
    print("Data types:")
    print(feature_data.schema)
else:
    print("\nFeature dataset not found!")

# 3. List raw data folders
print("\n=== RAW DATA FOLDERS ===")

for folder in sorted(RAW_DIR.iterdir()):
    if folder.is_dir():
        print(folder.name)

=== PROCESSED DATA ===
bank_quarter_features_v3.parquet | 171.92 MB
bank_quarter_panel_v3.parquet | 50.19 MB

=== FEATURE DATASET CHECK ===
Shape: (1663876, 36)
Columns: ['NCLNLS', 'LNATRES', 'ROA', 'REPDTE', 'ASSET', 'RBC1AAJ', 'CERT', 'NETINC', 'EQ', 'DEP', 'ID', 'REPORT_DATE', 'REPORT_Q_INDEX', 'FIRST_FAILURE_DATE', 'FAILURE_Q_INDEX', 'FAILURE_LABEL', 'IS_MERGER_CENSORED', 'FINAL_LABEL', 'YEAR', 'QUARTER', 'PREV_NETINC_YTD', 'PREV_QUARTER', 'QUARTERLY_NETINC', 'LOG_ASSET', 'EQUITY_TO_ASSETS', 'DEPOSITS_TO_ASSETS', 'NEGATIVE_NCLNLS_FLAG', 'NEGATIVE_LNATRES_FLAG', 'NPL_TO_ASSETS', 'ALLOWANCE_TO_ASSETS', 'ASSET_QOQ_GROWTH', 'EQUITY_RATIO_CHANGE', 'ROA_CHANGE', 'NPL_RATIO_CHANGE', 'ALLOWANCE_RATIO_CHANGE', 'EXTREME_ASSET_GROWTH_FLAG']
Data types:
Schema({'NCLNLS': Float64, 'LNATRES': Float64, 'ROA': Float64, 'REPDTE': String, 'ASSET': Int64, 'RBC1AAJ': Float64, 'CERT': Int64, 'NETINC': Float64, 'EQ': Float64, 'DEP': Float64, 'ID': String, 'REPORT_DATE': Date, 'REPORT_Q_INDEX': Int32, 'F

In [13]:
# Load the processed bank-quarter panel
panel_path = PROCESSED_DIR / "v3" / "bank_quarter_panel_v3.parquet"

panel_data = pl.read_parquet(panel_path)

# Inspect dataset
print("=== PANEL DATASET ===")
print("Shape:", panel_data.shape)
print("\nColumns:")
print(panel_data.columns)

print("\nData types:")
print(panel_data.schema)

# Check target variables
print("\n=== FAILURE LABEL DISTRIBUTION ===")

print(
    panel_data
    .group_by("FAILURE_LABEL")
    .len()
    .sort("FAILURE_LABEL")
)

print("\n=== FINAL LABEL DISTRIBUTION ===")

print(
    panel_data
    .group_by("FINAL_LABEL")
    .len()
    .sort("FINAL_LABEL")
)

# Check merger censoring
print("\n=== MERGER CENSORING ===")

print(
    panel_data
    .group_by("IS_MERGER_CENSORED")
    .len()
    .sort("IS_MERGER_CENSORED")
)

=== PANEL DATASET ===
Shape: (1663876, 18)

Columns:
['NCLNLS', 'LNATRES', 'ROA', 'REPDTE', 'ASSET', 'RBC1AAJ', 'CERT', 'NETINC', 'EQ', 'DEP', 'ID', 'REPORT_DATE', 'REPORT_Q_INDEX', 'FIRST_FAILURE_DATE', 'FAILURE_Q_INDEX', 'FAILURE_LABEL', 'IS_MERGER_CENSORED', 'FINAL_LABEL']

Data types:
Schema({'NCLNLS': Float64, 'LNATRES': Float64, 'ROA': Float64, 'REPDTE': String, 'ASSET': Int64, 'RBC1AAJ': Float64, 'CERT': Int64, 'NETINC': Float64, 'EQ': Float64, 'DEP': Float64, 'ID': String, 'REPORT_DATE': Date, 'REPORT_Q_INDEX': Int32, 'FIRST_FAILURE_DATE': Date, 'FAILURE_Q_INDEX': Int32, 'FAILURE_LABEL': Int8, 'IS_MERGER_CENSORED': Boolean, 'FINAL_LABEL': Int8})

=== FAILURE LABEL DISTRIBUTION ===
shape: (2, 2)
┌───────────────┬─────────┐
│ FAILURE_LABEL ┆ len     │
│ ---           ┆ ---     │
│ i8            ┆ u32     │
╞═══════════════╪═════════╡
│ 0             ┆ 1653016 │
│ 1             ┆ 10860   │
└───────────────┴─────────┘

=== FINAL LABEL DISTRIBUTION ===
shape: (3, 2)
┌─────────────┬─

In [14]:
# Display complete label distributions
print("=== FINAL LABEL COUNTS ===")

final_counts = (
    panel_data
    .group_by("FINAL_LABEL")
    .len()
    .sort("FINAL_LABEL", nulls_last=True)
)

for row in final_counts.iter_rows():
    print(f"FINAL_LABEL = {row[0]}: {row[1]:,} observations")

print("\n=== MERGER CENSORING COUNTS ===")

censor_counts = (
    panel_data
    .group_by("IS_MERGER_CENSORED")
    .len()
    .sort("IS_MERGER_CENSORED")
)

for row in censor_counts.iter_rows():
    print(f"IS_MERGER_CENSORED = {row[0]}: {row[1]:,} observations")

# Verify total observations
print("\nTotal observations:", panel_data.height)

=== FINAL LABEL COUNTS ===


FINAL_LABEL = 0: 1,597,219 observations
FINAL_LABEL = 1: 10,860 observations
FINAL_LABEL = None: 55,797 observations

=== MERGER CENSORING COUNTS ===
IS_MERGER_CENSORED = False: 1,608,079 observations
IS_MERGER_CENSORED = True: 55,797 observations

Total observations: 1663876


In [15]:
# Duplicate and data quality audit

print("DUPLICATE BANK-QUARTER CHECK")
print("----------------------------")

duplicate_count = (
    panel_data
    .group_by(["CERT", "REPORT_DATE"])
    .agg(pl.len().alias("ROW_COUNT"))
    .filter(pl.col("ROW_COUNT") > 1)
    .height
)

print("Duplicate CERT-REPORT_DATE groups:", duplicate_count)

print("\nDATE RANGE")
print("----------")
print("Start date:", panel_data["REPORT_DATE"].min())
print("End date:", panel_data["REPORT_DATE"].max())

print("\nMISSING VALUES")
print("--------------")

display(
    panel_data
    .select([
        pl.col(c).null_count().alias(c)
        for c in panel_data.columns
    ])
)

DUPLICATE BANK-QUARTER CHECK
----------------------------


Duplicate CERT-REPORT_DATE groups: 0

DATE RANGE
----------
Start date: 1984-03-31
End date: 2025-06-30

MISSING VALUES
--------------


NCLNLS,LNATRES,ROA,REPDTE,ASSET,RBC1AAJ,CERT,NETINC,EQ,DEP,ID,REPORT_DATE,REPORT_Q_INDEX,FIRST_FAILURE_DATE,FAILURE_Q_INDEX,FAILURE_LABEL,IS_MERGER_CENSORED,FINAL_LABEL
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
2499,4081,0,0,0,0,0,4624,4080,1,0,0,0,1575146,1575146,0,0,55797


## 4. Target Variable Construction and Validation

The final dataset contains 1,663,876 bank-quarter observations.

### 4.1 Failure Label Distribution

The initial failure label (`FAILURE_LABEL`) identifies whether a bank experiences a failure event within the next four quarters.

- Non-failure observations: 1,653,016
- Failure observations: 10,860

### 4.2 Merger-related Censoring

Observations whose prediction horizon overlaps with a qualifying merger event are treated as censored.

- Non-censored observations: 1,608,504
- Censored observations: 55,372

### 4.3 Final Label Distribution

After applying merger-related censoring:

| FINAL_LABEL | Observations | Meaning |
|---|---:|---|
| 0 | 1,597,644 | No failure event within the prediction horizon |
| 1 | 10,860 | Failure event within the prediction horizon |
| Null | 55,372 | Censored observations |

Censored observations are excluded from standard binary classification training.

**Validation:** The three final label groups sum to 1,663,876 observations, confirming that the label construction and censoring counts are internally consistent.

**Methodological note:** Treating qualifying merger events as censoring is a research assumption. It does not imply that every merger represents financial distress.